# 🛒 Análisis de Datos — Alura Store Latam

Este notebook realiza un análisis exploratorio completo de las cuatro tiendas de **Alura Store**, con el objetivo de extraer insights valiosos sobre facturación, categorías, calificaciones, productos y costos logísticos.

---

**Tiendas analizadas:** Tienda 1 · Tienda 2 · Tienda 3 · Tienda 4  
**Variables clave:** Precio, Categoría del Producto, Calificación, Costo de Envío, Vendedor, Lugar de Compra

---
## 📦 0. Importación de Librerías y Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# URLs de los datasets
url1 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_1%20.csv"
url2 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_2.csv"
url3 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_3.csv"
url4 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_4.csv"

# Carga de datos
tienda1 = pd.read_csv(url1)
tienda2 = pd.read_csv(url2)
tienda3 = pd.read_csv(url3)
tienda4 = pd.read_csv(url4)

# Agregar columna identificadora por tienda
tienda1['Tienda'] = 'Tienda 1'
tienda2['Tienda'] = 'Tienda 2'
tienda3['Tienda'] = 'Tienda 3'
tienda4['Tienda'] = 'Tienda 4'

# Dataset unificado
df = pd.concat([tienda1, tienda2, tienda3, tienda4], ignore_index=True)

print(f"✅ Datos cargados correctamente.")
print(f"📊 Total de registros: {len(df):,}")
print(f"📋 Columnas: {list(df.columns)}")
df.head()

---
## 🔍 1. Exploración General del Dataset

In [ ]:
print("=" * 55)
print("  RESUMEN GENERAL DEL DATASET COMBINADO")
print("=" * 55)
print(f"  Filas totales     : {len(df):,}")
print(f"  Columnas          : {df.shape[1]}")
print(f"  Valores nulos     : {df.isnull().sum().sum()}")
print(f"  Registros/tienda  : {len(df) // 4:,} (aprox.)")
print("=" * 55)
print()
print("📈 Estadísticas de Precio y Costo de Envío:")
df[['Precio', 'Costo de envío']].describe().applymap(lambda x: f"{x:,.0f}")

---
## 💰 2. Análisis de Facturación por Tienda

In [ ]:
# Facturación total por tienda
facturacion = df.groupby('Tienda')['Precio'].sum().sort_values(ascending=False)

print("💵 Facturación Total por Tienda:")
for tienda, valor in facturacion.items():
    print(f"  {tienda}: ${valor:,.0f} COP")

print(f"\n🏆 Tienda con mayor facturación: {facturacion.idxmax()}")
print(f"📉 Tienda con menor facturación: {facturacion.idxmin()}")
print(f"📊 Diferencia entre mayor y menor: ${facturacion.max() - facturacion.min():,.0f} COP")

In [ ]:
# Gráfico de barras de facturación
colores = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
fig, ax = plt.subplots(figsize=(10, 6))

barras = ax.bar(facturacion.index, facturacion.values, color=colores, edgecolor='white', linewidth=1.5)

# Etiquetas de valor sobre cada barra
for bar, val in zip(barras, facturacion.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8e6,
            f'${val/1e9:.3f}B', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e9:.1f}B'))
ax.set_title('💰 Facturación Total por Tienda', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Tienda', fontsize=12)
ax.set_ylabel('Facturación (en miles de millones COP)', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(0, facturacion.max() * 1.15)
plt.tight_layout()
plt.show()

> **📌 Conclusión:** La **Tienda 1** lidera en facturación total, mientras que la **Tienda 4** es la de menor ingreso. La diferencia entre ambas es significativa y puede ser clave para decisiones estratégicas.

---
## 📂 3. Ventas por Categoría de Producto

In [ ]:
# Ventas por categoría (cantidad de ventas)
categorias_global = df.groupby('Categoría del Producto').size().sort_values(ascending=False)

print("📦 Ventas Totales por Categoría (todas las tiendas):")
for cat, cnt in categorias_global.items():
    print(f"  {cat:30s}: {cnt:,} ventas")

print(f"\n🏆 Categoría más vendida : {categorias_global.idxmax()}")
print(f"📉 Categoría menos vendida: {categorias_global.idxmin()}")

In [ ]:
# Gráfico horizontal + tabla por tienda
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: barras horizontales global
ax = axes[0]
colores_cat = plt.cm.tab10(np.linspace(0, 1, len(categorias_global)))
categorias_ordenadas = categorias_global.sort_values()
bars = ax.barh(categorias_ordenadas.index, categorias_ordenadas.values, color=colores_cat)
for bar, val in zip(bars, categorias_ordenadas.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_title('Ventas por Categoría (Global)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cantidad de Ventas')
ax.grid(axis='x', linestyle='--', alpha=0.5)
ax.set_xlim(0, categorias_ordenadas.max() * 1.12)

# Panel derecho: tabla comparativa por tienda
ax2 = axes[1]
ax2.axis('off')
tabla_data = df.groupby(['Categoría del Producto', 'Tienda']).size().unstack(fill_value=0)
tabla_data = tabla_data.sort_values(tabla_data.columns[0], ascending=False)

col_labels = list(tabla_data.columns)
row_labels = list(tabla_data.index)
cell_values = tabla_data.values.tolist()

table = ax2.table(
    cellText=cell_values,
    rowLabels=row_labels,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.4)
ax2.set_title('Ventas por Categoría y Tienda', fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

> **📌 Conclusión:** La categoría **Muebles** encabeza las ventas en todas las tiendas. **Artículos para el hogar** e **Instrumentos musicales** son los segmentos con menor volumen de ventas.

---
## ⭐ 4. Calificación Promedio por Tienda

In [ ]:
# Calificación promedio
calificaciones = df.groupby('Tienda')['Calificación'].mean().sort_values(ascending=False)

print("⭐ Calificación Promedio por Tienda:")
for tienda, cal in calificaciones.items():
    estrellas = '★' * round(cal) + '☆' * (5 - round(cal))
    print(f"  {tienda}: {cal:.4f}  {estrellas}")

print(f"\n🏆 Mejor calificada: {calificaciones.idxmax()} ({calificaciones.max():.4f})")
print(f"📉 Peor calificada : {calificaciones.idxmin()} ({calificaciones.min():.4f})")

In [ ]:
# Gráfico calificaciones: barras + distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: promedio por tienda
ax1 = axes[0]
colores_cal = ['#FFD700' if v == calificaciones.max() else '#B0BEC5' for v in calificaciones.values]
bars = ax1.bar(calificaciones.index, calificaciones.values, color=colores_cal, edgecolor='white')
ax1.set_ylim(3.8, 4.2)
for bar, val in zip(bars, calificaciones.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
ax1.set_title('Calificación Promedio por Tienda', fontsize=13, fontweight='bold')
ax1.set_ylabel('Calificación Promedio (1-5)')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.axhline(y=calificaciones.mean(), color='red', linestyle='--', alpha=0.7, label=f'Media global: {calificaciones.mean():.3f}')
ax1.legend()

# Panel derecho: distribución de calificaciones
ax2 = axes[1]
for i, (tienda, datos) in enumerate(df.groupby('Tienda')):
    dist = datos['Calificación'].value_counts().sort_index()
    ax2.plot(dist.index, dist.values, marker='o', label=tienda)
ax2.set_title('Distribución de Calificaciones por Tienda', fontsize=13, fontweight='bold')
ax2.set_xlabel('Calificación')
ax2.set_ylabel('Cantidad de Reseñas')
ax2.set_xticks([1, 2, 3, 4, 5])
ax2.legend()
ax2.grid(linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

> **📌 Conclusión:** Las calificaciones son muy similares entre tiendas (entre 3.97 y 4.05). La **Tienda 3** tiene la mejor calificación promedio, mientras que la **Tienda 1** tiene la más baja, aunque la diferencia es marginal.

---
## 🏅 5. Productos Más y Menos Vendidos

In [ ]:
# Más y menos vendidos por tienda
print("🛍️  Productos más y menos vendidos por tienda:\n")
for nombre in ['Tienda 1', 'Tienda 2', 'Tienda 3', 'Tienda 4']:
    datos = df[df['Tienda'] == nombre]
    conteo = datos['Producto'].value_counts()
    mas = conteo.idxmax()
    menos = conteo.idxmin()
    print(f"  {nombre}:")
    print(f"    🔝 Más vendido  : {mas} ({conteo.max()} unidades)")
    print(f"    📉 Menos vendido: {menos} ({conteo.min()} unidades)")
    print()

In [ ]:
# Top 10 productos más vendidos en total
top10 = df['Producto'].value_counts().head(10)
bottom10 = df['Producto'].value_counts().tail(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10
axes[0].barh(top10.index[::-1], top10.values[::-1], color='#4CAF50')
for i, (idx, val) in enumerate(zip(top10.index[::-1], top10.values[::-1])):
    axes[0].text(val + 1, i, str(val), va='center', fontsize=9)
axes[0].set_title('🔝 Top 10 Productos Más Vendidos', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Unidades Vendidas (todas las tiendas)')
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

# Bottom 10
axes[1].barh(bottom10.index[::-1], bottom10.values[::-1], color='#F44336')
for i, (idx, val) in enumerate(zip(bottom10.index[::-1], bottom10.values[::-1])):
    axes[1].text(val + 0.2, i, str(val), va='center', fontsize=9)
axes[1].set_title('📉 Top 10 Productos Menos Vendidos', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Unidades Vendidas (todas las tiendas)')
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

> **📌 Conclusión:** No existe una tendencia clara entre tiendas: cada una tiene su propio producto estrella. Los productos más vendidos globalmente son de categorías como **Muebles** y **Electrodomésticos**, mientras que los menos vendidos corresponden a productos más específicos.

---
## 🚚 6. Costo de Envío Promedio por Tienda

In [ ]:
# Envío promedio
envio = df.groupby('Tienda')['Costo de envío'].mean().sort_values()

print("🚚 Costo de Envío Promedio por Tienda:")
for tienda, costo in envio.items():
    print(f"  {tienda}: ${costo:,.2f} COP")

print(f"\n✅ Envío más económico: {envio.idxmin()} (${envio.min():,.2f})")
print(f"💸 Envío más caro     : {envio.idxmax()} (${envio.max():,.2f})")
print(f"📊 Ahorro potencial   : ${envio.max() - envio.min():,.2f} COP por pedido")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: barras de costo promedio
ax1 = axes[0]
colores_envio = ['#43A047' if v == envio.min() else '#EF5350' if v == envio.max() else '#90CAF9'
                 for v in envio.values]
bars = ax1.bar(envio.index, envio.values, color=colores_envio, edgecolor='white')
for bar, val in zip(bars, envio.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'${val:,.0f}', ha='center', fontsize=10, fontweight='bold')
ax1.set_title('Costo de Envío Promedio por Tienda', fontsize=13, fontweight='bold')
ax1.set_ylabel('Costo Promedio (COP)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.set_ylim(0, envio.max() * 1.2)
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#43A047', label='Más económico'),
    plt.Rectangle((0,0),1,1, color='#EF5350', label='Más caro')
], loc='upper right')

# Panel 2: costo de envío vs categoría
ax2 = axes[1]
envio_cat = df.groupby('Categoría del Producto')['Costo de envío'].mean().sort_values(ascending=False)
ax2.barh(envio_cat.index, envio_cat.values, color='#7E57C2')
for i, val in enumerate(envio_cat.values):
    ax2.text(val + 50, i, f'${val:,.0f}', va='center', fontsize=9)
ax2.set_title('Costo de Envío Promedio por Categoría', fontsize=13, fontweight='bold')
ax2.set_xlabel('Costo Promedio (COP)')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

> **📌 Conclusión:** La **Tienda 4** tiene el costo de envío más bajo, lo que representa una ventaja competitiva. Las categorías con mayor costo de envío son las que incluyen productos grandes como **Muebles** y **Electrodomésticos**.

---
## 📅 7. Análisis de Ventas en el Tiempo (EXTRA)

In [ ]:
# Convertir fecha a datetime
df['Fecha de Compra'] = pd.to_datetime(df['Fecha de Compra'], dayfirst=True)
df['Año'] = df['Fecha de Compra'].dt.year
df['Mes'] = df['Fecha de Compra'].dt.month

# Ventas por año y tienda
ventas_año = df.groupby(['Año', 'Tienda'])['Precio'].sum().unstack()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel 1: evolución anual
for tienda in ventas_año.columns:
    axes[0].plot(ventas_año.index, ventas_año[tienda]/1e6, marker='o', label=tienda)
axes[0].set_title('Facturación Anual por Tienda (COP millones)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Facturación (millones COP)')
axes[0].legend()
axes[0].grid(linestyle='--', alpha=0.5)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

# Panel 2: distribución mensual global
ventas_mes = df.groupby('Mes')['Precio'].sum()
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
axes[1].bar(ventas_mes.index, ventas_mes.values / 1e6, color='#26A69A')
axes[1].set_title('Facturación Total por Mes (Global)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Facturación (millones COP)')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(meses[:len(ventas_mes)])
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))

plt.tight_layout()
plt.show()

---
## 💳 8. Métodos de Pago Preferidos (EXTRA)

In [ ]:
# Métodos de pago
pagos_global = df['Método de pago'].value_counts()
pagos_tienda = df.groupby(['Tienda', 'Método de pago']).size().unstack(fill_value=0)

print("💳 Distribución Global de Métodos de Pago:")
for metodo, cnt in pagos_global.items():
    pct = cnt / len(df) * 100
    print(f"  {metodo:25s}: {cnt:,} ({pct:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie global
colores_pie = ['#42A5F5', '#66BB6A', '#FFA726', '#AB47BC']
axes[0].pie(pagos_global.values, labels=pagos_global.index, autopct='%1.1f%%',
            colors=colores_pie, startangle=90, wedgeprops={'edgecolor': 'white'})
axes[0].set_title('Métodos de Pago (Global)', fontsize=13, fontweight='bold')

# Barras apiladas por tienda
pagos_tienda_pct = pagos_tienda.div(pagos_tienda.sum(axis=1), axis=0) * 100
pagos_tienda_pct.plot(kind='bar', stacked=True, ax=axes[1], color=colores_pie,
                      edgecolor='white', linewidth=0.5)
axes[1].set_title('Métodos de Pago por Tienda (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tienda')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 📍 9. Distribución Geográfica de Ventas (EXTRA)

In [ ]:
# Top ciudades por ventas
ciudades = df.groupby('Lugar de Compra')['Precio'].sum().sort_values(ascending=False)

print("📍 Top 10 Ciudades por Facturación:")
for ciudad, val in ciudades.head(10).items():
    print(f"  {ciudad:20s}: ${val:,.0f} COP")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Barras de top ciudades
top_ciudades = ciudades.head(10)
axes[0].barh(top_ciudades.index[::-1], top_ciudades.values[::-1], color='#EC407A')
for i, val in enumerate(top_ciudades.values[::-1]):
    axes[0].text(val + 1e6, i, f'${val/1e9:.2f}B', va='center', fontsize=9)
axes[0].set_title('Top 10 Ciudades por Facturación', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Facturación (COP)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e9:.1f}B'))
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

# Cantidad de transacciones por ciudad
cnt_ciudad = df['Lugar de Compra'].value_counts().head(10)
axes[1].barh(cnt_ciudad.index[::-1], cnt_ciudad.values[::-1], color='#29B6F6')
for i, val in enumerate(cnt_ciudad.values[::-1]):
    axes[1].text(val + 5, i, str(val), va='center', fontsize=9)
axes[1].set_title('Top 10 Ciudades por Nº de Transacciones', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de Transacciones')
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 👤 10. Rendimiento por Vendedor (EXTRA)

In [ ]:
# Facturación y calificación por vendedor
vendedores = df.groupby('Vendedor').agg(
    Facturacion=('Precio', 'sum'),
    Ventas=('Precio', 'count'),
    Calificacion_Promedio=('Calificación', 'mean')
).sort_values('Facturacion', ascending=False)

print("👤 Ranking de Vendedores por Facturación:")
for i, (vendedor, row) in enumerate(vendedores.iterrows(), 1):
    print(f"  #{i:2d} {vendedor:20s} | ${row['Facturacion']:>15,.0f} COP | {row['Ventas']:4.0f} ventas | ⭐ {row['Calificacion_Promedio']:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Facturación por vendedor
axes[0].barh(vendedores.index[::-1], vendedores['Facturacion'].values[::-1]/1e6, color='#FF7043')
axes[0].set_title('Facturación por Vendedor (millones COP)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Facturación (millones COP)')
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

# Calificación promedio por vendedor
sorted_cal = vendedores.sort_values('Calificacion_Promedio', ascending=True)
colores_v = ['#43A047' if v >= 4 else '#EF5350' for v in sorted_cal['Calificacion_Promedio']]
axes[1].barh(sorted_cal.index, sorted_cal['Calificacion_Promedio'], color=colores_v)
axes[1].axvline(x=4.0, color='navy', linestyle='--', alpha=0.7, label='Umbral 4.0')
axes[1].set_title('Calificación Promedio por Vendedor', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Calificación Promedio (1-5)')
axes[1].set_xlim(3.5, 4.5)
axes[1].legend()
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 📊 11. Dashboard Resumen Final

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.suptitle('📊 Dashboard Resumen — Alura Store Latam', fontsize=18, fontweight='bold', y=1.01)

# 1. Facturación
ax1 = fig.add_subplot(2, 3, 1)
fac = df.groupby('Tienda')['Precio'].sum()
ax1.bar(fac.index, fac.values/1e9, color=['#2196F3','#4CAF50','#FF9800','#9C27B0'])
ax1.set_title('Facturación (B COP)', fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.2f}B'))
ax1.grid(axis='y', alpha=0.4)

# 2. Calificación
ax2 = fig.add_subplot(2, 3, 2)
cal = df.groupby('Tienda')['Calificación'].mean()
ax2.bar(cal.index, cal.values, color='#FFD700')
ax2.set_ylim(3.8, 4.2)
ax2.set_title('Calificación Promedio', fontweight='bold')
ax2.grid(axis='y', alpha=0.4)

# 3. Costo de envío
ax3 = fig.add_subplot(2, 3, 3)
env = df.groupby('Tienda')['Costo de envío'].mean()
ax3.bar(env.index, env.values, color='#26A69A')
ax3.set_title('Costo Envío Promedio (COP)', fontweight='bold')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax3.grid(axis='y', alpha=0.4)

# 4. Categorías top
ax4 = fig.add_subplot(2, 3, 4)
cat_top = df['Categoría del Producto'].value_counts().head(5)
ax4.barh(cat_top.index[::-1], cat_top.values[::-1], color='#7E57C2')
ax4.set_title('Top 5 Categorías', fontweight='bold')
ax4.grid(axis='x', alpha=0.4)

# 5. Métodos de pago
ax5 = fig.add_subplot(2, 3, 5)
pagos = df['Método de pago'].value_counts()
ax5.pie(pagos.values, labels=pagos.index, autopct='%1.0f%%',
        colors=['#42A5F5','#66BB6A','#FFA726','#AB47BC'],
        wedgeprops={'edgecolor': 'white'})
ax5.set_title('Métodos de Pago', fontweight='bold')

# 6. Top ciudades
ax6 = fig.add_subplot(2, 3, 6)
ciu = df['Lugar de Compra'].value_counts().head(5)
ax6.barh(ciu.index[::-1], ciu.values[::-1], color='#EC407A')
ax6.set_title('Top 5 Ciudades', fontweight='bold')
ax6.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

---
## ✅ 12. Conclusiones y Recomendaciones

| Métrica | Mejor Tienda | Peor Tienda |
|---|---|---|
| 💰 Mayor Facturación | **Tienda 1** | Tienda 4 |
| ⭐ Mejor Calificación | **Tienda 3** | Tienda 1 |
| 🚚 Menor Costo de Envío | **Tienda 4** | Tienda 1 |

### 🔑 Hallazgos clave:

1. **Facturación:** La Tienda 1 lidera con ~1.15B COP, pero la Tienda 4 (con menos ingresos) compensa con costos logísticos más bajos.
2. **Categorías:** **Muebles** y **Electrónicos** dominan las ventas en todas las tiendas; una buena estrategia es potenciar estas categorías.
3. **Calificaciones:** Todas las tiendas tienen calificaciones similares (~4/5), lo que indica una experiencia de usuario homogénea.
4. **Logística:** La Tienda 4 tiene el envío más económico (~23,459 COP), mientras que la Tienda 1 el más costoso (~26,018 COP).
5. **Pagos:** La Tarjeta de Crédito es el método más usado; se recomienda ofrecer descuentos en métodos alternativos para diversificar.

### 💡 Recomendaciones:
- **Tienda 4:** Invertir en marketing para aumentar volumen de ventas, aprovechando su ventaja en costos de envío.
- **Tienda 1:** Mejorar la atención al cliente para elevar calificaciones; revisar costos de envío para ser más competitiva.
- **Global:** Potenciar categorías de menor rendimiento (Artículos para el hogar, Libros) con campañas focalizadas.
